# Streaming - Pipeline de Processamento

## Objetivo

Este notebook implementa o processamento em Streaming das novas medições
de alfabetização geradas pelo simulador.

O pipeline utiliza `trigger(availableNow=True)` devido às restrições do ambiente Databricks Serverless utilizado no projeto, que não suporta o trigger
`ProcessingTime`.

O `AvailableNow` mantém o processamento incremental do Spark Structured Streaming: a cada execução são consumidos apenas os novos eventos ainda não
registrados no checkpoint.

Dessa forma, o pipeline simula a ingestão em tempo quase real sem reprocessareventos já consumidos.

O fluxo implementado é:

`simulacao.eventos_alfabetizacao`
→ Bronze Streaming
→ Silver Streaming
→ Gold Streaming

# Configuração

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Configuração das Tabelas

In [0]:
tabela_origem = "simulacao.eventos_alfabetizacao"

tabela_bronze = "bronze.eventos_alfabetizacao_streaming"
tabela_silver = "silver.eventos_alfabetizacao_streaming"
tabela_gold = "gold.acompanhamento_metas_streaming"

## Função Parar Stream 

In [0]:
def parar_query_se_existir(nome_query):

    for query in spark.streams.active:

        if query.name == nome_query:

            print(f"Parando query existente: " f"{nome_query}")

            query.stop()

## Criando Volume

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS
workspace.simulacao.streaming_checkpoints
""")

## Checkpoints
O Spark Structured Streaming utiliza checkpoints para registrar o estado de processamento da pipeline.

O checkpoint permite que o Spark identifique quais dados já foram processados e continue a execução a partir do ponto correto em caso de
reinicialização.

Esse mecanismo evita o reprocessamento desnecessário dos mesmos eventos e fornece tolerância a falhas ao pipeline.

In [0]:
volume_checkpoints = "/Volumes/workspace/" "simulacao/streaming_checkpoints"

checkpoint_bronze = f"{volume_checkpoints}/bronze_alfabetizacao"

checkpoint_silver = f"{volume_checkpoints}/silver_alfabetizacao"

checkpoint_gold = f"{volume_checkpoints}/gold_alfabetizacao"

print(checkpoint_bronze)
print(checkpoint_silver)
print(checkpoint_gold)

# Bronze Streaming
A camada Bronze Streaming recebe os eventos da área de entrada sem aplicar transformações de negócio.

São adicionados apenas metadados técnicos relacionados à ingestão, preservando o conteúdo original do evento.

A leitura é realizada utilizando `readStream`, fazendo com que novos registros adicionados à origem sejam detectados e processados incrementalmente.

In [0]:
# Leitura com readStream
df_bronze_stream = spark.readStream.table(tabela_origem)

# Adicionando meta dados
df_bronze_stream = (
    df_bronze_stream.withColumn("_stream_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_system", F.lit("Simulador Tech Challenge"))
    .withColumn("_source_table", F.lit(tabela_origem))
)

## Persistindo a Bronze

Garantindo que duas querys iguais não ficarão rodando:

In [0]:
nome_query_bronze = "stream_bronze_alfabetizacao"

parar_query_se_existir(nome_query_bronze)

Iniciando o Stream

In [0]:
query_bronze = (
    df_bronze_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_bronze)
    .queryName(nome_query_bronze)
    .trigger(availableNow=True)
    .toTable(tabela_bronze)
)

query_bronze.awaitTermination()

## Processando Eventos Ocorridos

In [0]:
query_bronze.processAllAvailable()

Validando:

In [0]:
display(spark.table(tabela_bronze))

# Silver Streaming
A camada Silver Streaming é responsável pelo tratamento e validação dos eventos recebidos pela camada Bronze Streaming.

Nesta etapa, os eventos passam por regras de padronização e consistência antes de seguirem para consumo analítico.

Os eventos inválidos não são corrigidos ou preenchidos artificialmente, evitando a alteração do significado original dos dados.

Lendo com o `readStream`, mantendo o processamento incremental e permitindo que novos eventos que cheguem à camada Bronze sejam processados nas execuções
seguintes do pipeline.

In [0]:
df_silver_stream = spark.readStream.table(tabela_bronze)

print("DataFrame Streaming:", df_silver_stream.isStreaming)

## Padronização dos Campos
Os campos textuais utilizados para identificação e relacionamento são padronizados antes das validações.

In [0]:
df_silver_stream = (
    df_silver_stream
    # Trtando as colunas de nível geográfico, id geografia e rede
    .withColumn("nivel_geografico", F.upper(F.trim(F.col("nivel_geografico"))))
    .withColumn("id_geografia", F.trim(F.col("id_geografia")))
    .withColumn("rede", F.trim(F.col("rede")))
)

## Regras de Consistência

Antes da persistência na camada Silver, são aplicadas regras mínimas de consistência aos eventos.

Um evento é considerado válido quando:

- possui identificador do evento;
- possui timestamp do evento;
- possui ano de referência;
- possui nível geográfico válido;
- possui identificador geográfico;
- possui rede de ensino;
- a taxa de alfabetização está entre 0% e 100%;
- o percentual de participação está entre 0% e 100%, quando informado.

Os níveis geográficos aceitos são:

- `UF`;
- `MUNICIPIO`;
- `BRASIL`.

Essas validações evitam que eventos estruturalmente inválidos avancem no pipeline.

In [0]:
# Definição da regra
condicao_evento_valido = (
    F.col("event_id").isNotNull()
    & F.col("event_timestamp").isNotNull()
    & F.col("ano").isNotNull()
    & F.col("nivel_geografico").isin("UF", "MUNICIPIO", "BRASIL")
    & F.col("id_geografia").isNotNull()
    & F.col("rede").isNotNull()
    & F.col("taxa_alfabetizacao").between(0, 100)
    & (
        F.col("percentual_participacao").isNull()
        | F.col("percentual_participacao").between(0, 100)
    )
)

# Aplicando e filtrando apenas os registros válidos
df_silver_stream = (
    df_silver_stream
    .filter(condicao_evento_valido)
    
    # Inserindo metadado de processamento da silver
    .withColumn("_silver_processed_at", F.current_timestamp())
)

## Persistindo a Silver

Garantindo que duas querys iguais não ficarão rodando:

In [0]:
nome_query_silver = "stream_silver_alfabetizacao"

parar_query_se_existir(nome_query_silver)

Iniciando o Stream

In [0]:
query_silver = (
    df_silver_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_silver)
    .queryName(nome_query_silver)
    .trigger(availableNow=True)
    .toTable(tabela_silver)
)

query_silver.awaitTermination()

Validando

In [0]:
qtd_bronze = spark.table(tabela_bronze).count()
qtd_silver = spark.table(tabela_silver).count()

print(f"Eventos Bronze: {qtd_bronze}")
print(f"Eventos Silver: {qtd_silver}")

print(f"Eventos descartados: " f"{qtd_bronze - qtd_silver}")

# Gold Streaming

A Gold Streaming cruza as novas medições recebidas em Streaming com as metas históricas processadas no pipeline Batch.

A partir desse cruzamento são calculados o gap para a meta, o percentual de atingimento e o status do resultado.

## Preparação das Metas
Para cada território, rede e ano da meta, é utilizada a versão mais recente disponível.

In [0]:
# Carregando a base silver
df_metas_batch = spark.table("silver.metas")

# Window Function pra pegar a meta mais recente
window_meta = Window.partitionBy(
    "nivel_geografico", "id_geografia", "rede", "ano_meta"
).orderBy(F.desc("ano_referencia"))

# Criando Row Number e deixando os registros únicos
df_metas_recentes = (
    df_metas_batch
    .withColumn("_rn", F.row_number().over(window_meta))
    .filter(F.col("_rn") == 1)
    .select(
        "nivel_geografico",
        "id_geografia",
        "rede",
        "ano_meta",
        F.col("ano_referencia").alias("ano_referencia_meta"),
        "meta_alfabetizacao",
    )
)

## Leitura da Silver

Os eventos tratados são consumidos novamente como Streaming para serem enriquecidos com as metas Batch.

In [0]:
df_eventos_silver_stream = spark.readStream.table(tabela_silver)

print("Eventos são Streaming:", df_eventos_silver_stream.isStreaming)

print("Metas são Streaming:", df_metas_recentes.isStreaming)

## Integração Batch + Streaming

Os eventos são relacionados às metas por geografia, rede e ano.

A tabela de metas é pequena e, por isso, é utilizada como Broadcast Join.

In [0]:
df_gold_stream = (
    # Cruzando a tabela de metas com a de streams, por geografia, rede e ano da meta
    df_eventos_silver_stream.alias("evento")
    .join(
        F.broadcast(df_metas_recentes).alias("meta"),
        (F.col("evento.nivel_geografico") == F.col("meta.nivel_geografico"))
        & (F.col("evento.id_geografia") == F.col("meta.id_geografia"))
        & (F.col("evento.rede") == F.col("meta.rede"))
        & (F.col("evento.ano") == F.col("meta.ano_meta")),
        "left",
    )
    # Seleção dos Campos
    .select(
        F.col("evento.event_id"),
        F.col("evento.event_timestamp"),
        F.col("evento.tipo_evento"),
        F.col("evento.ano"),
        F.col("evento.nivel_geografico"),
        F.col("evento.id_geografia"),
        F.col("evento.rede"),
        F.col("evento.taxa_alfabetizacao"),
        F.col("evento.percentual_participacao"),
        F.col("meta.ano_referencia_meta"),
        F.col("meta.meta_alfabetizacao"),
    )

    # São calculados o gap para a meta
    .withColumn(
        "gap_meta_pp",
        F.round(F.col("taxa_alfabetizacao") - F.col("meta_alfabetizacao"), 2),
    )
    .withColumn(
        "percentual_atingimento_meta",
        F.when(
            F.col("meta_alfabetizacao") > 0,
            F.round(
                (F.col("taxa_alfabetizacao") / F.col("meta_alfabetizacao")) * 100, 2
            ),
        ),
    )
    # Cálculo do percentual de atingimento
    .withColumn(
        "flag_atingiu_meta",
        F.when(F.col("meta_alfabetizacao").isNull(), F.lit(None).cast("int"))
        .when(F.col("taxa_alfabetizacao") >= F.col("meta_alfabetizacao"), F.lit(1))
        .otherwise(F.lit(0)),
    )

    # Classificação do resultado
    .withColumn(
        "status_meta",
        F.when(F.col("meta_alfabetizacao").isNull(), "SEM META")
        .when(F.col("taxa_alfabetizacao") >= F.col("meta_alfabetizacao"), "ATINGIU A META")
        .otherwise("NÃO ATINGIU A META"),
    )

    # Metadados
    .withColumn("_gold_processed_at", F.current_timestamp())
)

## Persistindo a Gold

A Gold é persistida em Delta utilizando processamento incremental e checkpoint próprio.

Garantindo que duas querys iguais não ficarão rodando:

In [0]:
nome_query_gold = ("stream_gold_alfabetizacao")

parar_query_se_existir(nome_query_gold)

Iniciando o Stream

In [0]:
query_gold = (
    df_gold_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_gold)
    .queryName(nome_query_gold)
    .trigger(availableNow=True)
    .toTable(tabela_gold)
)

query_gold.awaitTermination()

## Validação

Validando a gold stream

In [0]:
df_gold_validacao = spark.table(tabela_gold)

display(
    df_gold_validacao.select(
        "ano",
        "nivel_geografico",
        "id_geografia",
        "rede",
        "taxa_alfabetizacao",
        "meta_alfabetizacao",
        "gap_meta_pp",
        "percentual_atingimento_meta",
        "flag_atingiu_meta",
        "status_meta",
    ).orderBy("nivel_geografico", "id_geografia")
)

Qtd. Registros

In [0]:
qtd_silver = spark.table(tabela_silver).count()

qtd_gold = spark.table(tabela_gold).count()

print(f"Eventos Silver: {qtd_silver}")

print(f"Eventos Gold: {qtd_gold}")

Comparando metas BATCH e STREAM

In [0]:
df_batch = (
    spark.table("gold.acompanhamento_metas")
    .select(
        "ano",
        "nivel_geografico",
        "id_geografia",
        "rede",
        "taxa_alfabetizacao",
        "meta_alfabetizacao",
        "gap_meta_pp",
        "percentual_atingimento_meta",
        "flag_atingiu_meta",
        "status_meta",
    )
    .withColumn("origem", F.lit("BATCH_OFICIAL"))
)

df_stream = (
    spark.table(tabela_gold)
    .select(
        "ano",
        "nivel_geografico",
        "id_geografia",
        "rede",
        "taxa_alfabetizacao",
        "meta_alfabetizacao",
        "gap_meta_pp",
        "percentual_atingimento_meta",
        "flag_atingiu_meta",
        "status_meta",
    )
    .withColumn("origem", F.lit("STREAMING_SIMULADO"))
)

df_comparativo = df_batch.unionByName(df_stream)


display(
    df_comparativo.filter(
        (F.col("nivel_geografico") == "UF") & (F.col("id_geografia") == "BA")
    ).orderBy("ano", "origem")
)

# Validação Incremental

Após a primeira execução, um novo evento foi publicado no simulador.

Ao executar novamente o pipeline, apenas o novo registro foi processado, elevando a quantidade total de eventos de 5 para 6.

Os eventos anteriores foram preservados sem duplicação, demonstrando o funcionamento dos checkpoints e do processamento incremental do Spark Structured Streaming.

In [0]:
%sql
select * from workspace.gold.acompanhamento_metas_streaming